# FFN spectrum — is there learned low-rank structure to exploit?

SwiGLU is ~40% of the params, so a low-rank / hypernet FFN is the biggest
structural lever left. cubby-lm's `ffn_compression/NOTES.md` says don't run that
bake-off blind: gate on whether the trained FFN's singular values have a **sharp
knee** (low-rank viable) or a **fat spectrum** (skip). This measures it on an
existing checkpoint — minutes, no training. Each trained matrix is compared to
its own init, since a decaying curve looks compressible no matter what.

In [ ]:
# --- setup: clone repo, deps, mount Drive (run once per session) ---
import os, subprocess, sys, time
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!pip -q install torch numpy sentencepiece
from google.colab import drive; drive.mount('/content/drive')
REPO = '/content/CubbyLLM'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- EDIT to your Drive locations ---
DRIVE     = '/content/drive/MyDrive/cubbyllm'
TOKENIZER = f'{DRIVE}/grillcheese_bbpe128k.json'   # your tokenizer .json/.model
CORPUS_DRIVE = f'{DRIVE}/token_cache'              # uint32 shards on Drive
CORPUS    = '/content/token_cache'                 # staged to LOCAL SSD

# Stage the corpus to local SSD. Drive-FUSE memmap reads bottleneck the GPU —
# this run measured 3,000 tok/s off Drive vs ~100,000 tok/s off local disk.
# Skip (comment out) if CORPUS is already populated this session.
!mkdir -p {CORPUS} && rsync -a --info=progress2 {CORPUS_DRIVE}/ {CORPUS}/
!du -sh {CORPUS}

In [ ]:
# --- run a script, tee to a log, catch a hung child on interrupt ---
def run(script, env_extra, log_name):
    env = dict(os.environ, CUBBY_SPM=TOKENIZER, CB_CORPUS=CORPUS, **env_extra)
    os.makedirs(f'{REPO}/validation/logs', exist_ok=True)
    log = f'{REPO}/validation/logs/{log_name}'
    p = None
    try:
        with open(log, 'a', encoding='utf-8', buffering=1) as f:
            f.write(f"\n=== {time.strftime('%F %T')} "
                    + ' '.join(f'{k}={v}' for k, v in sorted(env_extra.items())) + '\n')
            p = subprocess.Popen([sys.executable, '-u', f'validation/{script}'],
                                 cwd=REPO, env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end=''); f.write(line)
            p.wait()
    finally:
        if p and p.poll() is None:      # never leave a child holding the GPU
            p.terminate()

In [ ]:
run('exp_ffn_spectrum.py', dict(CB_CKPT=f'{DRIVE}/ab_mingru.pt'), 'ffn_spectrum.log')

### How to read
`r90/R` = fraction of directions holding 90% of the energy, **trained vs its own
init**. Fat spectrum (trained ≈ init, near full-rank) → skip the low-rank arm,
spend on ternary (bytes, ~free) or MoE. Sharp knee well below init → low-rank FFN
is worth a matched-budget A/B. Ternary is a *bytes* play, unaffected either way.